In [ ]:
# Seleção Final - Hall da Fama e GC Content
from Bio.SeqUtils import gc_fraction
from Bio import SeqIO

# --- CONFIGURAÇÃO ---
ARQUIVO_ELITE = "candidatos_elite.txt" # Sua lista de 422
ARQUIVO_FASTA = "../c_abscissum_data/ncbi_dataset/data/GCA_023376855.1/cds_from_genomic.fna"

# 1. HALL DA FAMA (Genes validados em literatura de RNAi antifúngico)
# Prioridade Máxima: Se o gene tiver esses nomes, ele sobe para o topo.
HALL_OF_FAME = [
    "chitin synthase", "chs",           # Parede celular (Alvo clássico)
    "beta-tubulin",                     # Citoesqueleto (Já validamos a segurança dele!)
    "elongation factor", "ef1",         # Tradução (Morte rápida)
    "vacuolar atpase", "v-atpase",      # Homeostase de pH
    "glyceraldehyde-3-phosphate", "gpd",# Glicólise (Energia)
    "hsp70", "heat shock",              # Estresse
    "scytalone", "polyketide synthase", # Patogenicidade (SCD)
    "pmk1", "map kinase"                # Sinalização de infecção
]

# 2. Critérios Químicos (Para síntese eficiente de dsRNA)
MIN_GC = 35.0  # Abaixo disso é instável
MAX_GC = 60.0  # Acima disso forma estruturas secundárias difíceis

# --- PROCESSAMENTO ---
print("--- INICIANDO RANKING FINAL ---")

# Carrega IDs da elite
ids_elite = set()
try:
    with open(ARQUIVO_ELITE, "r") as f:
        for linha in f:
            partes = linha.split("\t")
            if len(partes) > 0:
                ids_elite.add(partes[0])
except FileNotFoundError:
    print("ERRO: Arquivo 'candidatos_elite.txt' não encontrado.")

print(f"Analisando {len(ids_elite)} candidatos de elite...")

# Listas para classificação
platinum_candidates = [] # Estão no Hall da Fama + GC bom
gold_candidates = []     # GC bom, mas função genérica (ribossomo, etc)

for record in SeqIO.parse(ARQUIVO_FASTA, "fasta"):
    if record.id in ids_elite:
        desc = record.description.lower()
        seq = record.seq
        
        # Calcula GC
        gc = gc_fraction(seq) * 100
        
        # Filtro Químico
        if gc < MIN_GC or gc > MAX_GC:
            continue # Descarta genes com química ruim
            
        # Filtro Bibliográfico
        is_famous = any(target in desc for target in HALL_OF_FAME)
        
        item = {
            "id": record.id,
            "desc": record.description,
            "gc": gc,
            "len": len(seq)
        }
        
        if is_famous:
            platinum_candidates.append(item)
        else:
            gold_candidates.append(item)

# Ordenar por tamanho (genes muito pequenos < 200pb são ruins para RNAi, mas o BLAST já filtrou isso)
platinum_candidates.sort(key=lambda x: x['gc'], reverse=True)

# --- RELATÓRIO FINAL ---
print("\n" + "="*80)
print(f"TOP TIER - CANDIDATOS PLATINA (Hall da Fama + Química Perfeita)")
print("="*80)
print(f"Total encontrado: {len(platinum_candidates)}")

print(f"{'GENE ID':<35} | {'GC%':<5} | {'DESCRIÇÃO'}")
print("-" * 80)

for cand in platinum_candidates: # Mostra todos os platina
    desc_curta = " ".join(cand['desc'].split()[1:])[:50] # Remove ID e corta
    print(f"{cand['id'][:35]} | {cand['gc']:.1f} | {desc_curta}")

print("\n" + "="*80)
print(f"CANDIDATOS OURO (Bons substitutos)")
print("="*80)
print(f"Total encontrado: {len(gold_candidates)}")
# Mostra apenas os 5 melhores 'Ouro' como exemplo
for cand in gold_candidates[:5]:
    desc_curta = " ".join(cand['desc'].split()[1:])[:50]
    print(f"{cand['id'][:35]} | {cand['gc']:.1f} | {desc_curta}")

# Salvar lista finalíssima
with open("TOP_ALVOS_FINAIS.txt", "w") as f:
    f.write("CATEGORIA\tID\tGC_CONTENT\tDESCRICAO\n")
    for c in platinum_candidates:
        f.write(f"PLATINUM\t{c['id']}\t{c['gc']:.1f}\t{c['desc']}\n")
    for c in gold_candidates:
        f.write(f"GOLD\t{c['id']}\t{c['gc']:.1f}\t{c['desc']}\n")

print("\nLista final salva em 'TOP_ALVOS_FINAIS.txt'.")

--- INICIANDO RANKING FINAL ---
Analisando 422 candidatos de elite...

TOP TIER - CANDIDATOS PLATINA (Hall da Fama + Química Perfeita)
Total encontrado: 19
GENE ID                             | GC%   | DESCRIÇÃO
--------------------------------------------------------------------------------
lcl|SDAQ01000083.1_cds_KAI3541540.1 | 59.0 | [locus_tag=CABS02_10714] [protein=translation elon
lcl|SDAQ01000016.1_cds_KAI3555617.1 | 58.7 | [locus_tag=CABS02_03993] [protein=scytalone dehydr
lcl|SDAQ01000074.1_cds_KAI3543065.1 | 58.4 | [locus_tag=CABS02_10189] [protein=translation elon
lcl|SDAQ01000020.1_cds_KAI3555019.1 | 56.8 | [locus_tag=CABS02_04858] [protein=PKSN polyketide 
lcl|SDAQ01000125.1_cds_KAI3536226.1 | 56.3 | [locus_tag=CABS02_12601] [protein=stress-activated
lcl|SDAQ01000100.1_cds_KAI3539126.1 | 55.6 | [locus_tag=CABS02_11547] [protein=chitin synthase 
lcl|SDAQ01000038.1_cds_KAI3551757.1 | 55.5 | [locus_tag=CABS02_07203] [protein=translation elon
lcl|SDAQ01000001.1_cds_KAI3559674.1

### O "Dream Team" (Top 5 Selecionados), segundo GEMINI:

O Alvo de Patogenicidade (SCD): **lcl|SDAQ01000016.1_cds_KAI3555617.1_3992**

Função: Síntese de melanina. Sem isso, o fungo não penetra na laranja.

Vantagem: Não mata o fungo no solo, apenas impede a infecção. Altíssima especificidade.

O Alvo Metabólico (EF-1): lcl|SDAQ01000074.1_cds_KAI3543065.1_10219

Função: Fator de Elongação 1 (Síntese Proteica).

Vantagem: Essencial. Silenciar isso colapsa a produção de proteínas do fungo. Morte celular rápida.

O Alvo Estrutural (Chitin Synthase): lcl|SDAQ01000110.1_cds_KAI3537946.1_11987

Função: Síntese de Quitina (Parede Celular).

Vantagem: Sem quitina, a célula do fungo "explode" devido à pressão osmótica. Alvo clássico de fungicidas, agora via RNAi.

O Alvo Mitocondrial (EF-Tu): lcl|SDAQ01000083.1_cds_KAI3541540.1_10744

Função: Tradução de proteínas dentro da mitocôndria.

Vantagem: Ataca a "usina de energia" do fungo. Mecanismo distinto do EF-1 citoplasmático.

O Alvo de Virulência Secundária (PKSN): lcl|SDAQ01000020.1_cds_KAI3555019.1_4857

Função: Polyketide Synthase (Biossíntese de toxinas/metabólitos).

Vantagem: Muitas vezes associada à supressão do sistema imune da planta.